<a href="https://colab.research.google.com/github/smosharof/Resume.Walkthrough/blob/main/BERT_Compare_Documents.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install PyPDF2 transformers torch scikit-learn

In [ ]:
import PyPDF2
from transformers import BertTokenizer, BertModel
import torch
from sklearn.metrics.pairwise import cosine_similarity

def extract_text_from_pdf(pdf_path):
    """Extracts text from a PDF file."""
    text = ""
    with open(pdf_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        for page in reader.pages:
            text += page.extract_text() or ""
    return text

def chunk_text(text, chunk_size=512, chunk_overlap=100):
    """Splits text into smaller chunks."""
    words = text.split()
    chunks = []
    for i in range(0, len(words), chunk_size - chunk_overlap):
        chunk = " ".join(words[i:i + chunk_size])
        chunks.append(chunk)
    return chunks

def get_bert_embeddings(text_list, model, tokenizer):
    """Gets BERT embeddings for a list of texts."""

    encoded_input = tokenizer.batch_encode_plus(
        text_list,
        padding=True,
        truncation=True,
        max_length=512,
        return_tensors='pt'
    )
    input_ids = encoded_input['input_ids']
    attention_mask = encoded_input['attention_mask']

    with torch.no_grad():
        model_output = model(input_ids, attention_mask=attention_mask)
    # Use the CLS token embeddings (the first token)
    embeddings = model_output.last_hidden_state[:, 0, :]  # Shape: [batch_size, 768]
    return embeddings

# --- Main ---

# 1. Extract text from the PDFs
apple_text = extract_text_from_pdf("Apple 10K Q4 2024.pdf")
meta_text = extract_text_from_pdf("Meta 10K Q4 2024.pdf")

# 2. Chunk the text
apple_chunks = chunk_text(apple_text)
meta_chunks = chunk_text(meta_text)

# 3. Initialize BERT model and tokenizer
model_name = 'bert-base-uncased'  # You can experiment with other models
tokenizer = BertTokenizer.from_pretrained(model_name)
model = BertModel.from_pretrained(model_name)

# 4. Get embeddings
apple_embeddings = get_bert_embeddings(apple_chunks, model, tokenizer)
meta_embeddings = get_bert_embeddings(meta_chunks, model, tokenizer)

# 5. Calculate similarity (compare each chunk from Apple to each chunk from Meta)
similarity_matrix = cosine_similarity(apple_embeddings, meta_embeddings)

# 6. Analyze the similarity matrix
# For example, find the most similar chunks:
import numpy as np

max_similarity = np.max(similarity_matrix)
max_row_idx, max_col_idx = np.unravel_index(np.argmax(similarity_matrix), similarity_matrix.shape)

print(f"Maximum similarity: {max_similarity:.4f}")
print(f"Apple chunk index: {max_row_idx}, Meta chunk index: {max_col_idx}")
print("\nApple Chunk:\n", apple_chunks[max_row_idx])
print("\nMeta Chunk:\n", meta_chunks[max_col_idx])

# Further analysis: You could calculate average similarity, find chunks above a threshold, etc.